# Xử lý ngôn ngữ tự nhiên - CS221.Q21.KHTN

## Demo chương 7: Large language model

### Nhóm 1:
- Bảo Quý Định Tân - 24520028
- Lê Văn Thức - 24521748
- Lê Phạm Thành Nhân - 24520022

### Ở demo cho chương LLM này ta sẽ sử dụng lại bài toán ở demo của chương 4 trên dữ liệu SA-Hotel

In [ ]:
!pip install -q transformers torch scikit-learn pyvi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 69.1 MB/s eta 0:00:00


In [ ]:
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score
from pyvi import ViTokenizer
import numpy as np
import warnings

warnings.filterwarnings('ignore')


In [ ]:
# ==========================================
# 1. Configuration & Setup
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {DEVICE}")

MODEL_NAME = "vinai/phobert-base-v2"
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 2e-5

# Aspect Configuration
entities = ["HOTEL", "ROOMS", "ROOM_AMENITIES", "FACILITIES", "SERVICE", "LOCATION", "FOOD&DRINKS"]
attributes = ["GENERAL", "PRICES", "DESIGN&FEATURES", "CLEANLINESS", "COMFORT", "QUALITY", "STYLE&OPTIONS", "MISCELLANEOUS"]
all_aspects = sorted([f"{e}#{a}" for e in entities for a in attributes])
NUM_ASPECTS = len(all_aspects)

# Label Mapping
label_to_id = {'null': 0, 'positive': 1, 'negative': 2, 'neutral': 3}
id_to_label = {0: 'null', 1: 'positive', 2: 'negative', 3: 'neutral'}

Running on device: cuda


In [ ]:
# ==========================================
# 2. Data Loading & Preprocessing (Google Drive)
# ==========================================
from google.colab import drive

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

def load_data(filepath):
    texts, labels = [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.read().strip().split('\n')

    for i in range(0, len(lines), 4):
        if i + 2 >= len(lines): break
        # PhoBERT requires word segmentation for optimal performance
        text = ViTokenizer.tokenize(lines[i+1].strip())
        label_line = lines[i+2].strip()

        aspect_dict = {}
        if label_line:
            matches = re.findall(r'\{([^,]+),\s*([^}]+)\}', label_line)
            for aspect, polarity in matches:
                aspect_dict[aspect.strip()] = polarity.strip()

        texts.append(text)
        labels.append(aspect_dict)
    return texts, labels

print("\nLoading data from Google Drive...")

# 2. Define the base path to your specific NLP folder
base_path = '/content/drive/MyDrive/UIT/NLP/VLSP2018-SA-train-dev-test/'

# 3. Load the data using the new paths
train_texts, train_labels = load_data(base_path + '1-VLSP2018-SA-Hotel-train (7-3-2018).txt')
dev_texts, dev_labels = load_data(base_path + '2-VLSP2018-SA-Hotel-dev (7-3-2018).txt')
test_texts, test_labels = load_data(base_path + '3-VLSP2018-SA-Hotel-test (8-3-2018).txt')

# combined_train_texts = train_texts + dev_texts
# combined_train_labels = train_labels + dev_labels

class ABSADataset(Dataset):
    def __init__(self, texts, labels_dicts, tokenizer):
        self.texts = texts
        self.tokenizer = tokenizer

        # Convert dictionary labels to a matrix of shape (N_samples, 56_aspects)
        self.labels = []
        for labels_dict in labels_dicts:
            row = [label_to_id.get(labels_dict.get(aspect, 'null'), 0) for aspect in all_aspects]
            self.labels.append(row)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=MAX_LEN,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': self.labels[idx]
        }

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_loader = DataLoader(
    ABSADataset(train_texts, train_labels, tokenizer),
    batch_size=BATCH_SIZE,
    shuffle=True
)
test_loader = DataLoader(ABSADataset(test_texts, test_labels, tokenizer), batch_size=BATCH_SIZE)

Mounting Google Drive...
Mounted at /content/drive

Loading data from Google Drive...


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
train_texts[4]
# train_labels[2]

for i in range(0, 10):
  print(train_texts[i], end='\n')
  print(train_labels[i], end='\n\n')

Rộng_rãi KS mới nhưng rất vắng . Các dịch_vụ chất_lượng chưa cao và thiếu .
{'HOTEL#DESIGN&FEATURES': 'positive', 'HOTEL#GENERAL': 'negative'}

Địa_điểm thuận_tiện , trong vòng bán_kính 1,5 km nhiều quán ăn ngon
{'LOCATION#GENERAL': 'positive'}

Phục_vụ , view đẹp , vị_trí
{'SERVICE#GENERAL': 'positive', 'HOTEL#GENERAL': 'positive', 'LOCATION#GENERAL': 'positive'}

thuận_tiện , sạch_sẽ , vui_vẻ hài_lòng
{'HOTEL#COMFORT': 'positive', 'HOTEL#CLEANLINESS': 'positive', 'SERVICE#GENERAL': 'positive'}

Vị_trí đẹp ; Có quán bar view đẹp ; Nhân_viên thân_thiện
{'LOCATION#GENERAL': 'positive', 'FACILITIES#GENERAL': 'positive', 'SERVICE#GENERAL': 'positive'}

- Co view huong Ho_tay - sach se - nhan vien tan tinh
{'HOTEL#GENERAL': 'positive', 'HOTEL#CLEANLINESS': 'positive', 'SERVICE#GENERAL': 'positive'}

Phòng_ốc sạch , giường thoải_mái , nhân_viên thân_thiện .
{'ROOMS#CLEANLINESS': 'positive', 'ROOM_AMENITIES#COMFORT': 'positive', 'SERVICE#GENERAL': 'positive'}

gần Hồ_Tây , view nhìn ra hồ lã

In [ ]:
# ==========================================
# 3. Multi-Task PhoBERT Model
# ==========================================
class MultiTaskPhoBERT(nn.Module):
    def __init__(self, n_aspects, n_classes):
        super().__init__()
        self.phobert = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(0.2)
        # Create 56 separate classification heads (one for each aspect)
        self.classifiers = nn.ModuleList([nn.Linear(768, n_classes) for _ in range(n_aspects)])

    def forward(self, input_ids, attention_mask):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        # Extract the <s> token (CLS equivalent) representation
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)

        # Get logits for all 56 aspects
        logits = [classifier(pooled_output) for classifier in self.classifiers]
        # Stack into shape: (batch_size, n_aspects, n_classes)
        return torch.stack(logits, dim=1)

model = MultiTaskPhoBERT(n_aspects=NUM_ASPECTS, n_classes=4).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

In [ ]:
# ==========================================
# 4. Training Loop
# ==========================================
print("\nStarting Training on GPU...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE) # Shape: (batch, 56)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask) # Shape: (batch, 56, 4)

        # Flatten tensors to easily compute loss across all aspects at once
        loss = criterion(logits.view(-1, 4), labels.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {total_loss/len(train_loader):.4f}")


Starting Training on GPU...
Epoch 1/10 | Train Loss: 0.7875
Epoch 2/10 | Train Loss: 0.4216
Epoch 3/10 | Train Loss: 0.3118
Epoch 4/10 | Train Loss: 0.2681
Epoch 5/10 | Train Loss: 0.2414
Epoch 6/10 | Train Loss: 0.2214
Epoch 7/10 | Train Loss: 0.2045
Epoch 8/10 | Train Loss: 0.1897
Epoch 9/10 | Train Loss: 0.1767
Epoch 10/10 | Train Loss: 0.1649


In [ ]:
# ==========================================
# 5. Evaluation on Test Set
# ==========================================
print("\nEvaluating on Test Set...")
model.eval()
all_preds = []
all_trues = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].cpu().numpy()

        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=2).cpu().numpy()

        all_preds.extend(preds)
        all_trues.extend(labels)

all_preds = np.array(all_preds)
all_trues = np.array(all_trues)

print(f"{'='*60}")
print(f"{'ASPECT':<30} | {'ACCURACY':<10} | {'MACRO F1':<10}")
print(f"{'-'*60}")

acc_list, f1_list = [], []

for idx, aspect in enumerate(all_aspects):
    y_true_aspect = all_trues[:, idx]
    y_pred_aspect = all_preds[:, idx]

    # Check if this aspect actually appears in the test set (ignore all-'null' aspects)
    if len(set(y_true_aspect)) > 1 or 0 not in set(y_true_aspect):
        acc = accuracy_score(y_true_aspect, y_pred_aspect)
        f1 = f1_score(y_true_aspect, y_pred_aspect, average='macro', zero_division=0)

        acc_list.append(acc)
        f1_list.append(f1)
        print(f"{aspect:<30} | {acc:.4f}     | {f1:.4f}")

print(f"{'='*60}")
if acc_list:
    print(f"{'TRUNG BÌNH TỔNG THỂ':<30} | {sum(acc_list)/len(acc_list):.4f}     | {sum(f1_list)/len(f1_list):.4f}")


Evaluating on Test Set...
ASPECT                         | ACCURACY   | MACRO F1  
------------------------------------------------------------
FACILITIES#CLEANLINESS         | 0.9917     | 0.3319
FACILITIES#COMFORT             | 0.9567     | 0.3260
FACILITIES#DESIGN&FEATURES     | 0.8967     | 0.2819
FACILITIES#GENERAL             | 0.9650     | 0.2455
FACILITIES#MISCELLANEOUS       | 0.9867     | 0.3311
FACILITIES#PRICES              | 0.9783     | 0.2473
FACILITIES#QUALITY             | 0.9150     | 0.2389
FOOD&DRINKS#MISCELLANEOUS      | 0.9950     | 0.3325
FOOD&DRINKS#PRICES             | 0.9850     | 0.3308
FOOD&DRINKS#QUALITY            | 0.8933     | 0.4356
FOOD&DRINKS#STYLE&OPTIONS      | 0.8383     | 0.4003
HOTEL#CLEANLINESS              | 0.9350     | 0.5413
HOTEL#COMFORT                  | 0.9050     | 0.4168
HOTEL#DESIGN&FEATURES          | 0.9117     | 0.3905
HOTEL#GENERAL                  | 0.8850     | 0.4244
HOTEL#MISCELLANEOUS            | 0.8867     | 0.2350
HOTEL#P